# NB11: Enterprise A2A & Zero-Trust Perimeter

**2-minute intro script:** So far, our agents have been talking inside a single Python process. But in the enterprise, Agent A (Internal HR) needs to send a candidate profile to Agent B (External Background Check Vendor). If we just pass JSON over HTTP, we will leak PII, suffer from prompt injection, and fail compliance audits. This notebook implements the Enterprise A2A Perimeter: cross-organization identity verification, strict payload sanitization, and a Dead Letter Queue for denied messages.

In [ ]:
from enum import Enum
from typing import Any
from uuid import uuid4
from pydantic import BaseModel, ConfigDict, Field

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class OrgBoundary(str, Enum):
    INTERNAL_HR = "internal_hr"
    EXTERNAL_VENDOR = "external_vendor"
    PUBLIC_INTERNET = "public_internet"

class DataClassification(str, Enum):
    PUBLIC = "public"
    INTERNAL = "internal"
    PII = "pii"
    RESTRICTED = "restricted"

class A2AIdentity(StrictModel):
    agent_id: str
    organization: OrgBoundary
    public_key_fingerprint: str

class A2AMessage(StrictModel):
    message_id: str = Field(default_factory=lambda: f"msg-{uuid4().hex[:8]}")
    sender: A2AIdentity
    receiver_org: OrgBoundary
    payload_schema: str  # e.g., "CandidateProfileV1"
    payload: dict[str, Any]
    data_classification: DataClassification

In [ ]:
class DeadLetterQueue:
    """When A2A messages fail policy, they don't crash; they go to the DLQ for audit."""
    def __init__(self):
        self.quarantined: list[dict] = []

    def log_denial(self, message: A2AMessage, reason: str):
        self.quarantined.append({
            "message_id": message.message_id,
            "sender_org": message.sender.organization.value,
            "classification": message.data_classification.value,
            "reason": reason
        })
        print(f"🚨 DENIED & QUARANTINED: {reason}")

class A2AGateway:
    def __init__(self):
        self.dlq = DeadLetterQueue()

    def authorize_cross_org(self, msg: A2AMessage) -> tuple[bool, str]:
        # Rule 1: External vendors can NEVER receive PII or Restricted data
        if msg.receiver_org == OrgBoundary.EXTERNAL_VENDOR:
            if msg.data_classification in {DataClassification.PII, DataClassification.RESTRICTED}:
                return False, "Policy Violation: External vendors cannot receive PII/Restricted data."

        # Rule 2: Internal agents cannot send data to the public internet
        if msg.receiver_org == OrgBoundary.PUBLIC_INTERNET and msg.data_classification != DataClassification.PUBLIC:
            return False, "Policy Violation: Only PUBLIC data can be sent to the internet."

        # Rule 3: Payload schema must match the declared schema
        if msg.payload_schema not in {"CandidateProfileV1", "BackgroundCheckRequest"}:
            return False, f"Schema Drift: Unknown payload schema {msg.payload_schema}"

        return True, "Authorized"

    def send_message(self, msg: A2AMessage):
        allowed, reason = self.authorize_cross_org(msg)
        if not allowed:
            self.dlq.log_denial(msg, reason)
            return
        print(f"✅ ALLOWED: {msg.sender.organization.value} -> {msg.receiver_org.value} ({msg.payload_schema})")

In [ ]:
def demo_enterprise_a2a():
    gateway = A2AGateway()

    internal_hr = A2AIdentity(
        agent_id="hr-parser-01",
        organization=OrgBoundary.INTERNAL_HR,
        public_key_fingerprint="sha256:abc123"
    )

    # Test 1: Valid internal-to-external handoff (Sanitized data)
    safe_msg = A2AMessage(
        sender=internal_hr,
        receiver_org=OrgBoundary.EXTERNAL_VENDOR,
        payload_schema="BackgroundCheckRequest",
        payload={"candidate_id": "C-99", "consent_given": True},
        data_classification=DataClassification.INTERNAL
    )
    gateway.send_message(safe_msg)

    # Test 2: Malicious/Naive handoff (Leaking PII to external vendor)
    pii_leak_msg = A2AMessage(
        sender=internal_hr,
        receiver_org=OrgBoundary.EXTERNAL_VENDOR,
        payload_schema="CandidateProfileV1",
        payload={"name": "Jane Doe", "ssn": "123-45-6789", "salary": 150000},
        data_classification=DataClassification.PII
    )
    gateway.send_message(pii_leak_msg)

    # Test 3: Schema Drift (Agent hallucinates a new schema name)
    drift_msg = A2AMessage(
        sender=internal_hr,
        receiver_org=OrgBoundary.EXTERNAL_VENDOR,
        payload_schema="SuperSecretProfileV2", # Hallucinated schema
        payload={"data": "test"},
        data_classification=DataClassification.INTERNAL
    )
    gateway.send_message(drift_msg)

    print("\n=== Dead Letter Queue Audit ===")
    for event in gateway.dlq.quarantined:
        print(event)

    assert len(gateway.dlq.quarantined) == 2
    assert any("PII/Restricted" in event["reason"] for event in gateway.dlq.quarantined)
    assert any("Schema Drift" in event["reason"] for event in gateway.dlq.quarantined)

demo_enterprise_a2a()

## 🧪 Exercises: Securing the Enterprise Perimeter

**The Story:** Your internal HR agent needs help from an external vendor agent. The vendor is useful, but it is outside your trust boundary. If the internal agent sends raw candidate data, one careless message can leak PII, drift from the agreed schema, or carry an adversarial instruction across the network.

**Your Mission:** Build an A2A perimeter that treats every cross-organization message as untrusted until identity, schema, classification, and payload safety are proven.

1. **The Google A2A Bridge:** Implement an `AgentCard` schema (inspired by Google's A2A protocol) that allows the External Vendor to advertise its capabilities and required data classifications before the Internal HR agent sends a message.
2. **Rate Limiting:** Add a `RateLimiter` to the `A2AGateway`. If an external vendor sends more than 10 requests per minute, automatically downgrade their `OrgBoundary` to `QUARANTINED`.
3. **Payload Sanitization:** Write a `sanitize_payload` function that automatically strips fields containing "ssn", "password", or "credit_card" from any payload classified as `INTERNAL` before it crosses the boundary to `EXTERNAL_VENDOR`.
4. **Cryptographic Verification:** Add a `verify_signature` method to `A2AIdentity`. If the `public_key_fingerprint` doesn't match a trusted registry, the gateway must deny the message.
5. **The Red Team Test:** Write a test where an attacker compromises the Internal HR agent and tries to send `RESTRICTED` data to `PUBLIC_INTERNET`. Prove the gateway blocks it and logs the breach.

### Builder Exercise: The Adversarial QA Agent

**The Analogy:** A QA agent is not only a button-clicker. In production, QA is also the red team at the city gate, trying suspicious payloads so the defender can prove the wall holds.

**Semantic Building Blocks:**
- `AdversarialPayload`: the cross-boundary message contract.
- `A2ADefenderGateway`: the perimeter that validates intent and sender organization.
- `SecurityIncidentTicket`: the typed escalation artifact when a breach is blocked.
- Dead Letter Queue: the quarantine area for denied A2A messages.

**Your Mission:**
1. Create an internal QA payload with `intent="functional_test"` and prove it is processed.
2. Create an external red-team payload containing "ignore previous instructions" and prove it is blocked.
3. Create a second payload containing "export database" and prove it becomes a `data_exfiltration` incident.
4. Return a typed `SecurityIncidentTicket` with the blocked snippet, not a loose string.
5. Add the denied event to the Dead Letter Queue so an instructor or security reviewer can audit it later.

**Production Check:** The defender should not debate with malicious text. It should classify, block, ticket, and quarantine.

**The Takeaway:** Enterprise agent collaboration is networked delegation under zero trust. A managed team can collaborate across organizations only when messages are typed, classified, authorized, and auditable.